# Conditional 3D scale-up

Generate a larger categorical volume with tiled L-MPDD sampling and scale guidance. Base-size anchor images are centered on their requested absolute output planes.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
import time

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app.api import AnchorSlice, PredictOptions
from src.app.runtime import (
    build_dataset,
    build_loader,
    load_defaults,
    load_predict_config,
    load_predictor,
)
from src.modeling.phases import quantize_phase


def take_slice(volume, axis, index):
    return np.take(volume, index, axis=axis)


## Parameters

`AnchorSlice.index` is the absolute plane index in the large output. The anchor image is a centered patch on that plane. Large L-MPDD keeps the diffusion schedule length, so runtime rises sharply with volume size; an RTX 2060 took about 10 minutes for a minimal 80³ regression, while the full 128³ quality sweep exceeded 30 minutes.

In [ ]:
SCALE_SIZE = 128
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
SAVE_RESULT = True
USE_SAVED_RESULT = False
METHOD_VERSION = "latent-scale-v4"

## Predict

Scale-up uses one shared 3D latent residual. Diffusion and descriptor losses run on balanced overlapping latent crops, and scale anchors use sampled regions from the exact final tri-axis decoder without copying labels. The final optimization result is returned directly, followed by optional one-pass categorical Refine. `scale.decode_batch_size: null` decodes each stage in one batch on a large-memory GPU.

In [ ]:
model_runs, predict_config = load_predict_config(PREDICT_CONFIG)
model_runs = {
    name: None
    if path is None
    else Path(path) if Path(path).is_absolute() else ROOT / path
    for name, path in model_runs.items()
}
result_path = model_runs["diffusion_run_dir"] / "predictions" / f"conditional_lmpdd_{SCALE_SIZE}.npz"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = Namespace(**load_defaults(model_runs["vae_run_dir"] / "vae.yaml"))
args.data_dir = ROOT / args.data_dir
args.batch_size = 16

batch, _ = next(
    build_loader(build_dataset(args), args, device=torch.device("cpu"))
)
target_images = [
    quantize_phase(image, args.num_phases).numpy()
    for image in batch[:, 0]
]
phase_counts = np.bincount(
    np.stack(target_images).reshape(-1),
    minlength=args.num_phases,
)
reference_phase_fractions = tuple((phase_counts / phase_counts.sum()).tolist())
anchor_image = target_images[0]
base_size = anchor_image.shape[0]
anchor_index = SCALE_SIZE // 2
condition_start = (SCALE_SIZE - base_size) // 2
anchors = [
    AnchorSlice(image=anchor_image, axis=0, index=anchor_index),
    AnchorSlice(image=anchor_image.copy(), axis=1, index=anchor_index),
]
if predict_config["phase_fractions"] is None:
    predict_config["phase_fractions"] = reference_phase_fractions
options = PredictOptions(
    num_phases=args.num_phases,
    **predict_config,
)
phase_fractions = np.asarray(options.phase_fractions, dtype=np.float32)

if USE_SAVED_RESULT:
    with np.load(result_path, allow_pickle=False) as saved:
        if str(saved["method_version"]) != METHOD_VERSION:
            raise ValueError("saved result uses an older scale-up method")
        if tuple(saved["volume"].shape) != (SCALE_SIZE,) * 3:
            raise ValueError("saved volume shape does not match SCALE_SIZE")
        if not np.array_equal(saved["anchor_axes"], [anchor.axis for anchor in anchors]):
            raise ValueError("saved anchor axes do not match current anchors")
        if not np.array_equal(saved["anchor_indices"], [anchor.index for anchor in anchors]):
            raise ValueError("saved anchor indices do not match current anchors")
        if not np.allclose(saved["phase_fractions"], phase_fractions):
            raise ValueError("saved phase fractions do not match current options")
        volume_np = saved["volume"].copy()
        stats = {
            key.removeprefix("stat__"): torch.from_numpy(saved[key].copy())
            for key in saved.files
            if key.startswith("stat__")
        }
        scale_steps = int(saved["scale_steps"])
        elapsed_seconds = float(saved["elapsed_seconds"])
    volume = torch.from_numpy(volume_np)
    print("loaded:", result_path)
else:
    predictor = load_predictor(**model_runs, device=device)
    start_time = time.perf_counter()
    volume, stats = predictor.predict(options, anchors=anchors, volume_size=SCALE_SIZE)
    elapsed_seconds = time.perf_counter() - start_time
    volume_np = volume.cpu().numpy()
    scale_history = stats["scale_history"]
    scale_steps = int(scale_history["step"].numel())

    if SAVE_RESULT:
        result_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "volume": volume_np,
            "anchor_axes": np.array([anchor.axis for anchor in anchors], dtype=np.int64),
            "anchor_indices": np.array([anchor.index for anchor in anchors], dtype=np.int64),
            "phase_fractions": phase_fractions,
            "elapsed_seconds": np.array(elapsed_seconds, dtype=np.float64),
            "scale_steps": np.array(scale_steps, dtype=np.int64),
            "method_version": np.array(METHOD_VERSION),
        }
        payload.update({
            f"stat__{key}": value.detach().cpu().numpy()
            for key, value in stats.items()
            if torch.is_tensor(value)
        })
        np.savez_compressed(result_path, **payload)
        print("saved:", result_path)

print("device:", device)
print(f"elapsed: {elapsed_seconds:.1f} seconds")
print("volume:", volume_np.shape, volume.dtype)
print("anchor patch start:", condition_start)
print("absolute anchors:", [(anchor.axis, anchor.index) for anchor in anchors])
print("scale guidance steps:", scale_steps)
print("refine applied:", bool(stats["refine_applied"]))
print("anchor mismatches:", np.round(stats["final_anchor_mismatches"].cpu().numpy(), 4).tolist())
print("volume phase fraction:", np.round(stats["final_phase_fraction"].cpu().numpy(), 4).tolist())

## Final diagnostics

Inspect the final scale-up result without candidate selection, calibration, or heuristic pass/fail gates. The topology values are diagnostics, not targets inferred from 2D references.

In [ ]:
anchor_mismatches = []
generated_patches = []
for anchor in anchors:
    plane = take_slice(volume_np, anchor.axis, anchor.index)
    generated = plane[
        condition_start : condition_start + base_size,
        condition_start : condition_start + base_size,
    ]
    generated_patches.append(generated)
    anchor_mismatches.append(np.mean(generated != anchor.image))

anchor_mismatches = np.asarray(anchor_mismatches)
volume_phase_fraction = stats["final_phase_fraction"].cpu().numpy()
axis_transition_rate = stats["final_axis_transition_rate"].cpu().numpy()
axis_run_mae = stats["final_axis_run_profile_mae"].cpu().numpy()
axis_repeat_rate = stats["final_axis_exact_repeat_rate"].cpu().numpy()
axis_near_repeat_rate = stats["final_axis_near_repeat_rate"].cpu().numpy()
axis_repeat_streak = stats["final_axis_max_repeat_streak"].cpu().numpy()
axis_boundary_jump = stats["final_axis_global_boundary_jump"].cpu().numpy()
component_count = stats["final_component_count"].cpu().numpy()
euler_3d_density = stats["final_euler_3d_density"].cpu().numpy()
phase_axis_percolation = stats["final_phase_axis_percolation"].cpu().numpy()
print("target phase fraction:", phase_fractions.tolist())
print("volume phase fraction:", np.round(volume_phase_fraction, 4).tolist())
print("axis transition rate:", np.round(axis_transition_rate, 4).tolist())
print("axis run-profile MAE:", np.round(axis_run_mae, 4).tolist())
print("exact repeated-slice rate:", np.round(axis_repeat_rate, 4).tolist())
print("near-duplicate slice rate:", np.round(axis_near_repeat_rate, 4).tolist())
print("maximum repeat streak:", axis_repeat_streak.astype(int).tolist())
print("global boundary jump:", np.round(axis_boundary_jump, 4).tolist())
print("3D component count by phase:", component_count.astype(int).tolist())
print("3D Euler density by phase:", np.round(euler_3d_density, 6).tolist())
print("phase/axis percolation:", phase_axis_percolation.astype(int).tolist())

fig, axes = plt.subplots(len(anchors), 3, figsize=(11, 3.6 * len(anchors)), squeeze=False)
for row, (anchor, generated) in enumerate(zip(anchors, generated_patches)):
    difference = generated != anchor.image
    panels = [
        (anchor.image, f"condition axis={anchor.axis}", "gray", 0, args.num_phases - 1),
        (generated, "generated center patch", "gray", 0, args.num_phases - 1),
        (difference, f"different {anchor_mismatches[row]:.1%}", "magma", 0, 1),
    ]
    for axis, (image, title, cmap, vmin, vmax) in zip(axes[row], panels):
        axis.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        axis.set_title(title)
        axis.axis("off")
plt.tight_layout()

indices = np.linspace(0, SCALE_SIZE - 1, 9).round().astype(int)
fig, axes = plt.subplots(3, len(indices), figsize=(18, 6), squeeze=False)
for row, axis_id in enumerate(range(3)):
    for column, index in enumerate(indices):
        axes[row, column].imshow(
            take_slice(volume_np, axis_id, index),
            cmap="gray",
            vmin=0,
            vmax=args.num_phases - 1,
            interpolation="nearest",
        )
        axes[row, column].set_title(f"axis {axis_id} / {index}")
        axes[row, column].axis("off")
plt.tight_layout()

## 3D slice browser

Choose an axis and move the slider to inspect every generated slice.

In [ ]:
axis_selector = widgets.ToggleButtons(
    options=[("XY", 0), ("XZ", 1), ("YZ", 2)],
    value=0,
    description="axis",
)
index_slider = widgets.IntSlider(
    value=volume_np.shape[0] // 2,
    min=0,
    max=volume_np.shape[0] - 1,
    step=1,
    description="index",
    continuous_update=False,
)


def show_slice(axis, index):
    plt.figure(figsize=(5, 5))
    plt.imshow(
        take_slice(volume_np, axis, index),
        cmap="gray",
        vmin=0,
        vmax=args.num_phases - 1,
        interpolation="nearest",
    )
    plt.title(f"axis {axis} / index {index}")
    plt.axis("off")
    plt.show()


slice_output = widgets.interactive_output(
    show_slice,
    {"axis": axis_selector, "index": index_slider},
)
display(widgets.HBox([axis_selector, index_slider]), slice_output)
